# Tutorial 7: Measuring and Exporting

Detection tells you *where* colonies are. Measurement tells you *what they
are* — how big, how round, how bright. In this tutorial you will add
measurements to a pipeline, extract a DataFrame of colony features, and
export the results for downstream analysis.

**What you will learn:**

1. Add measurement operations to a pipeline
2. Use `pipeline.apply_and_measure()` to get a DataFrame
3. Understand the output columns
4. Export to CSV and Parquet

## Imports

In [1]:
import phenotypic as pht
from phenotypic.data import load_yeast_plate
from phenotypic.enhance import BlurGauss, EnhanceLocalContrast
from phenotypic.detect import OtsuDetector
from phenotypic.measure import MeasureSize, MeasureShape, MeasureIntensity

## Build a Pipeline with Measurements

The `meas` parameter accepts a list of measurement operations. Each one
extracts a different set of features from the detected colonies.

In [2]:
plate = load_yeast_plate()

pipeline = pht.ImagePipeline(
    ops=[BlurGauss(sigma=2.0), EnhanceLocalContrast(clip_limit=0.01), OtsuDetector()],
    meas=[MeasureSize(), MeasureShape(), MeasureIntensity()],
)

## Apply and Measure

`.apply_and_measure()` runs the full pipeline (enhance → detect → measure)
and returns a [pandas DataFrame](https://pandas.pydata.org/docs/user_guide/dsintro.html)
with one row per detected colony.

In [3]:
df = pipeline.apply_and_measure(plate)
print(f"Measured {len(df)} colonies across {df.shape[1]} features")
df.head()

Measured 7 colonies across 53 features


,Size_Area,Size_IntegratedIntensity,Size_Perimeter,Size_ConvexArea,Size_BboxArea,Size_MajorAxisLength,Size_MinorAxisLength,Size_MinFeretDiameter,Size_MaxFeretDiameter,Size_InscribedRadius,...,Bbox_MaxRR,Bbox_MaxCC,Bbox_IntensityWeightedCenterRR,Bbox_IntensityWeightedCenterCC,Bbox_DistWeightedCenterRR,Bbox_DistWeightedCenterCC,Grid_RowNum,Grid_ColNum,Grid_RowMajorIdx,Grid_ColMajorIdx
0,22670.0,13244.210029,569.085353,22703.0,29920.0,174.202696,166.040896,166.762587,176.161857,79.881162,...,278,277,188.628120,190.642286,189.115912,190.522931,0,0,0,0
1,17055.0,10056.952051,493.487373,17105.5,22197.0,149.136304,145.879057,144.800000,151.347283,70.064256,...,258,1077,180.521263,1003.831491,180.830685,1003.171207,0,2,2,4
2,16047.0,9216.498028,471.345238,15959.5,20448.0,145.886830,140.078299,139.300036,146.348898,69.814039,...,252,1481,179.111878,1409.794201,179.506605,1409.024055,0,3,3,6
3,13329.0,7522.484134,431.889394,13292.5,17028.0,132.892445,127.815190,125.262360,133.416641,62.008064,...,250,664,184.258571,599.483198,184.551175,599.125678,0,1,1,2
4,17939.0,10945.558373,500.801082,17887.5,23250.0,153.653217,148.819114,147.557575,154.301653,72.835431,...,665,1487,586.998752,1410.776659,587.048721,1410.124326,1,3,7,7


## Explore the Columns

Each measurement operation contributes its own set of columns. Let's see
what we got.

In [4]:
print("All columns:")
for col in df.columns:
    print(f"  {col}")

All columns:
  Size_Area
  Size_IntegratedIntensity
  Size_Perimeter
  Size_ConvexArea
  Size_BboxArea
  Size_MajorAxisLength
  Size_MinorAxisLength
  Size_MinFeretDiameter
  Size_MaxFeretDiameter
  Size_InscribedRadius
  Size_MedianRadius
  Size_MeanRadius
  Size_RobustMeanRadius
  Size_MaxRadius
  Shape_Circularity
  Shape_Eccentricity
  Shape_Solidity
  Shape_Extent
  Shape_Compactness
  Shape_Orientation
  Shape_MeanBoundaryDist
  Shape_MedianBoundaryDist
  Intensity_IntegratedIntensity
  Intensity_MinimumIntensity
  Intensity_MaximumIntensity
  Intensity_MeanIntensity
  Intensity_MedianIntensity
  Intensity_StandardDeviationIntensity
  Intensity_CoefficientVarianceIntensity
  Intensity_LowerQuartileIntensity
  Intensity_UpperQuartileIntensity
  Intensity_InterquartileRangeIntensity
  Intensity_Density
  Intensity_ConvexDensity
  Metadata_ImageName
  Metadata_ImageType
  Metadata_BitDepth
  Metadata_FileSuffix
  Object_Label
  Bbox_CenterRR
  Bbox_CenterCC
  Bbox_MinRR
  Bbox_MinCC

Here are the highlights from each measurement:

**MeasureSize:**
- `Size_Area` — colony size in pixels
- `Size_IntegratedIntensity` — sum of grayscale pixel values
- `Size_MajorAxisLength` / `Size_MinorAxisLength` — fitted ellipse axes
- `Size_RobustMeanRadius` / `Size_MaxRadius` — typical radius of the colony body, and the reach of its farthest edge (a large gap flags a runner or spur)

**MeasureShape:**
- `Shape_Circularity` — how round the colony is (1.0 = perfect circle)
- `Shape_Solidity` — ratio of colony area to convex hull area
- `Shape_Eccentricity` — elongation (0 = circular, approaching 1 = elongated)

**MeasureIntensity:**
- `Intensity_MeanIntensity` / `Intensity_MedianIntensity` — average colony brightness
- `Intensity_StandardDeviationIntensity` — variation within the colony
- `Intensity_MinimumIntensity` / `Intensity_MaximumIntensity` — intensity extremes

## Quick Statistics

Since the result is a standard pandas DataFrame, you can use all the usual
pandas methods to explore it.

In [5]:
df[["Size_Area", "Shape_Circularity", "Intensity_MeanIntensity"]].describe()

,Size_Area,Shape_Circularity,Intensity_MeanIntensity
count,7.000000,7.000000,7.000000
mean,16869.857143,0.894006,0.582158
std,3076.517210,0.010850,0.015325
min,13329.000000,0.879643,0.564370
25%,14984.500000,0.885698,0.571459
50%,17055.000000,0.897971,0.583767
75%,17533.000000,0.900684,0.586948
max,22670.000000,0.907665,0.610154


## Export to CSV

For sharing with collaborators or importing into spreadsheet software,
export to CSV.

In [6]:
df.to_csv("colony_measurements.csv")
print("Saved to colony_measurements.csv")

Saved to colony_measurements.csv


## Export to Parquet

For large datasets, [Parquet](https://parquet.apache.org/) is more
efficient — it is compressed, preserves column types, and loads much
faster than CSV.

In [7]:
df.to_parquet("colony_measurements.parquet")
print("Saved to colony_measurements.parquet")

Saved to colony_measurements.parquet


## Clean Up

In [8]:
import os
os.remove("colony_measurements.csv")
os.remove("colony_measurements.parquet")

## Summary

You have extracted colony features and exported them for analysis:

- **`meas=[MeasureSize(), MeasureShape(), MeasureIntensity()]`** — add measurements to a pipeline
- **`pipeline.apply_and_measure(plate)`** — run the full pipeline and get a DataFrame
- **`.to_csv()`** / **`.to_parquet()`** — export for downstream tools

The result is a standard pandas DataFrame, so you can filter, group, plot,
and analyze it with any tool in the Python ecosystem.

**Next up:** [Tutorial 8: Using Prefab Pipelines](08_using_prefab_pipelines.ipynb) —
discover PhenoTypic's pre-built pipelines for common organisms and plate types.